# EV Usage Profiles — Overview

Visualises **all** EV usage profiles in this directory (`data/ev_usage_profiles/`).

The headline chart is the same **EV charging schedule** timeline used in the
cross-year combined data plot
(`plotting/out/data_plots/cross_year_combined/all_years_01_Jan_combined.html`).
It is produced by reusing the exact loader and figure builder behind that plot
(`plotting/data_plotting/figure_builders.py:build_ev_schedule_figure`), so the
styling and behaviour match one-to-one — just over **every** profile here
(training `ev_N` *and* held-out `ev_eval_N`) instead of the config-selected subset.

Each profile is one charging point: rows with data are **plug-in** events and the
following empty row marks **departure** (see `README.md`).

## Setup

Locate the repo root (so the `plotting` package is importable regardless of where
Jupyter is launched) and import the shared loader / figure builder.

In [1]:
# Auto-reload edited project modules (e.g. plotting.data_plotting.figure_builders)
# so re-running cells picks up source changes without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from *start* (cwd by default) until the AdvBuildingGym root."""
    start = Path(start or Path.cwd()).resolve()
    for cand in (start, *start.parents):
        if (cand / "pyproject.toml").exists() and (cand / "plotting").is_dir():
            return cand
    raise RuntimeError("Could not locate the AdvBuildingGym repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reuse the exact loader + figure builder behind the cross-year combined plot
# so this chart matches plotting/out/data_plots/.../*_combined.html.
from plotting.data_plotting.loaders import load_profiles
from plotting.data_plotting.figure_builders import build_ev_schedule_figure
from plotting.utils import write_figure_list_html

EV_DIR = REPO_ROOT / "data" / "ev_usage_profiles"
print("Repo root:       ", REPO_ROOT)
print("EV profiles dir: ", EV_DIR)

Repo root:        /hkfs/home/haicore/iai/dj0397/AdvBuildingGym
EV profiles dir:  /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/data/ev_usage_profiles


## Load every profile

`load_profiles` reads each single-day CSV and adds a `minutes` column (minutes
since midnight) that serves as the common 24-hour x-axis. It returns
`{file_stem: DataFrame}`.

In [2]:
# Every EV usage profile in the directory: training (ev_N) + held-out eval (ev_eval_N).
profile_files = sorted(p.name for p in EV_DIR.glob("ev_*.csv"))
print(f"Found {len(profile_files)} profiles:", profile_files)

profiles = load_profiles(EV_DIR, profile_files, timestamp_col="start")
list(profiles)

Found 10 profiles: ['ev_0.csv', 'ev_1.csv', 'ev_2.csv', 'ev_3.csv', 'ev_4.csv', 'ev_5.csv', 'ev_eval_0.csv', 'ev_eval_1.csv', 'ev_eval_2.csv', 'ev_eval_3.csv']


['ev_0',
 'ev_1',
 'ev_2',
 'ev_3',
 'ev_4',
 'ev_5',
 'ev_eval_0',
 'ev_eval_1',
 'ev_eval_2',
 'ev_eval_3']

## Session summary table

Flatten every plug-in session across all profiles into one tidy table, pairing
each connection with its departure exactly as the figure builder does (a row
with `max_cap_kWh` set is a plug-in; the following empty row — or end of day —
is departure). Profiles with no plug-in (e.g. an always-idle baseline) yield no
rows here.

In [3]:
def _hhmm(minutes: float) -> str:
    return f"{int(minutes) // 60:02d}:{int(minutes) % 60:02d}"


def summarise_sessions(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Flatten every plug-in session across all profiles into one table.

    Mirrors the connection/departure pairing in build_ev_schedule_figure.
    """
    records: list[dict] = []
    for label, df in frames.items():
        rows = df.sort_values("minutes").reset_index(drop=True)
        i = 0
        while i < len(rows):
            row = rows.iloc[i]
            if pd.isna(row.get("max_cap_kWh")):
                i += 1
                continue
            plug_in = float(row["minutes"])
            depart = 1440.0
            if i + 1 < len(rows) and pd.isna(rows.iloc[i + 1].get("max_cap_kWh")):
                depart = float(rows.iloc[i + 1]["minutes"])
                i += 1  # consume the departure row
            records.append({
                "profile": label,
                "plug_in": _hhmm(plug_in),
                "departure": _hhmm(depart),
                "connected_h": round((depart - plug_in) / 60.0, 2),
                "max_cap_kWh": row.get("max_cap_kWh"),
                "max_charging_kW": row.get("max_charging_kW"),
                "v2g_enabled": row.get("v2g_enabled"),
                "start_soc": row.get("start_soc"),
                "target_soc": row.get("target_soc"),
                "target_soc_reach_duration_h": row.get("target_soc_reach_duration_h"),
            })
            i += 1
    return pd.DataFrame.from_records(records)


sessions = summarise_sessions(profiles)
print(f"{len(sessions)} charging sessions across {sessions['profile'].nunique()} profiles")
sessions

19 charging sessions across 9 profiles


,profile,plug_in,departure,connected_h,max_cap_kWh,max_charging_kW,v2g_enabled,start_soc,target_soc,target_soc_reach_duration_h
0,ev_1,07:30,16:00,8.50,50.0,6.6,False,0.55,0.80,8.00
1,ev_1,18:00,24:00,6.00,77.0,11.0,True,0.45,0.80,10.00
2,ev_2,00:00,07:00,7.00,75.0,11.0,True,0.50,0.95,6.00
3,ev_2,08:00,16:00,8.00,88.0,11.0,True,0.70,0.95,6.00
4,ev_2,17:30,24:00,6.50,56.0,6.6,False,0.50,0.90,8.00
5,ev_3,00:00,06:30,6.50,77.0,11.0,True,0.43,0.81,6.25
6,ev_3,21:00,24:00,3.00,77.0,11.0,True,0.32,0.73,4.00
7,ev_4,12:00,18:00,6.00,56.0,6.6,False,0.35,0.85,6.00
8,ev_4,19:00,24:00,5.00,77.0,11.0,True,0.38,0.90,8.00
9,ev_5,00:00,07:30,7.50,78.0,11.0,True,0.65,0.95,5.00


## EV charging schedule

Two timeline bar charts — **train** (`ev_N`) and held-out **eval** (`ev_eval_N`)
— one horizontal lane per profile, each plug-in window drawn as a bar annotated
with its `start_soc → target_soc` transition plus the EV's max charging power and
battery capacity. Hover a bar for V2G and target duration. The always-idle
`ev_0` profile has no plug-in events and appears as an empty lane in the train
chart. Identical styling to the cross-year combined data plot.

In [4]:
# Split into train (ev_N, incl. the empty ev_0 lane) and held-out eval (ev_eval_N),
# preserving order.
train_profiles = {k: v for k, v in profiles.items() if "eval" not in k}
eval_profiles = {k: v for k, v in profiles.items() if "eval" in k}
print("train:", list(train_profiles), "| eval:", list(eval_profiles))

train_fig = build_ev_schedule_figure(
    train_profiles, title="")
eval_fig = build_ev_schedule_figure(
    eval_profiles, title="")
train_fig.show()
eval_fig.show()

train: ['ev_0', 'ev_1', 'ev_2', 'ev_3', 'ev_4', 'ev_5'] | eval: ['ev_eval_0', 'ev_eval_1', 'ev_eval_2', 'ev_eval_3']


## Export

Write static **PDF**s (one per chart, via kaleido) and a combined interactive
**HTML** next to this notebook. The HTML uses the same writer that produces the
`*_combined.html` data plots; the PDFs are exported *first* because that writer
clears each figure's fixed width/height for its responsive card layout.

In [5]:
# Static PDFs first: write_figure_list_html strips each figure's width/height
# for its responsive cards, so export images before that call. Kaleido needs a
# Chrome binary (fetched once into ~/.cache on headless nodes).
from plotting.utils import ensure_chrome_for_kaleido

ensure_chrome_for_kaleido()
for fig, stem in [(train_fig, "ev_schedule_train"), (eval_fig, "ev_schedule_eval")]:
    pdf_path = EV_DIR / f"{stem}.pdf"
    fig.write_image(str(pdf_path))
    print("Wrote", pdf_path)

# Combined interactive HTML (same writer as the *_combined.html data plots).
out_html = EV_DIR / "ev_schedule_all_profiles.html"
write_figure_list_html([train_fig, eval_fig], str(out_html))
print("Wrote", out_html)

Wrote /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/data/ev_usage_profiles/ev_schedule_train.pdf
Wrote /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/data/ev_usage_profiles/ev_schedule_eval.pdf
Wrote /hkfs/home/haicore/iai/dj0397/AdvBuildingGym/data/ev_usage_profiles/ev_schedule_all_profiles.html
